# Experiment 2: Naïve Bayes and Decision Tree Classification (Fine-Tuned on 365-Day Dataset)

**Objective:** Develop, fine-tune, and compare Naïve Bayes and Decision Tree classifiers to predict whether tennis will be played (`PlayTennis`) based on weather conditions and seasonal features (`Month`, `Outlook`, `Temperature`, `Humidity`, `Wind`).

**Dataset:** Play Tennis Dataset (365 Days across January–December)
**Attributes:** `Day`, `Date`, `Month`, `Outlook`, `Temperature`, `Humidity`, `Wind`, `PlayTennis`

---

## Table of Contents
1. [Setup & Configuration](#1-setup--configuration)
2. [Data Loading & Exploration](#2-data-loading--exploration)
3. [Data Preprocessing & Feature Encoding](#3-data-preprocessing--feature-encoding)
4. [Train/Test Split](#4-traintest-split)
5. [Model Training & Fine-Tuning — Naïve Bayes](#5-model-training--fine-tuning--naïve-bayes)
6. [Model Training & Fine-Tuning — Decision Tree](#6-model-training--fine-tuning--decision-tree)
7. [Model Evaluation](#7-model-evaluation)
8. [Confusion Matrices](#8-confusion-matrices)
9. [ROC Curves](#9-roc-curves)
10. [Decision Tree Visualization & Rules](#10-decision-tree-visualization--rules)
11. [Feature Importance Analysis](#11-feature-importance-analysis)
12. [Cross-Validation Analysis](#12-cross-validation-analysis)
13. [Depth Sweep — Overfitting Analysis](#13-depth-sweep--overfitting-analysis)
14. [Misclassification Analysis](#14-misclassification-analysis)
15. [Model Comparison Report](#15-model-comparison-report)
16. [Observations & Conclusions](#16-observations--conclusions)

---
## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # Use non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

# Add project root to sys.path
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Project modules
from src.data_loader      import load_classification_data, check_data_quality, augment_data
from src.feature_encoding import encode_categorical_features, split_and_prepare_data, save_processed_data
from src.model_training   import (train_naive_bayes, tune_naive_bayes, train_categorical_naive_bayes,
                                   train_decision_tree, tune_decision_tree, save_model, depth_sweep)
from src.evaluation       import (evaluate_classification, cross_validate_model,
                                   generate_roc_data, compare_models,
                                   extract_tree_rules, analyze_misclassifications)
from src.visualization    import (plot_confusion_matrices, plot_roc_curves,
                                   plot_decision_tree, plot_feature_importance,
                                   plot_depth_sweep, plot_cv_comparison)

# Load config
CONFIG_PATH = os.path.join(PROJECT_ROOT, 'config', 'parameters.json')
with open(CONFIG_PATH) as f:
    CFG = json.load(f)

def P(rel): return os.path.join(PROJECT_ROOT, rel)

print('✔ All modules imported successfully.')
print(f'   Project root : {PROJECT_ROOT}')
print(f'   Config loaded: {CONFIG_PATH}')
print()
print('Configuration:')
print(json.dumps(CFG, indent=2))

---
## 2. Data Loading & Exploration

In [ ]:
raw_path = P(CFG['data']['raw_path'])
drop_cols = CFG['data']['drop_columns']
TARGET    = CFG['data']['target_column']

# Load the 365-day dataset
df_raw = load_classification_data(raw_path, drop_cols=drop_cols)
print(f'Shape: {df_raw.shape}')
df_raw.head(10)

In [ ]:
# Data quality check
quality = check_data_quality(df_raw, TARGET)

In [ ]:
# Visualise Target Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

vc = df_raw[TARGET].value_counts()
colors = ['#4C72B0', '#DD8452']

# Bar chart
axes[0].bar(vc.index, vc.values, color=colors, edgecolor='white', width=0.5)
for i, (cls, cnt) in enumerate(vc.items()):
    axes[0].text(i, cnt + 3, str(cnt), ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Class Distribution — PlayTennis (365 Days)', fontweight='bold')
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, max(vc.values) * 1.15)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Pie chart
axes[1].pie(vc.values, labels=vc.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Proportion', fontweight='bold')

plt.tight_layout()
os.makedirs(P('results'), exist_ok=True)
plt.savefig(P('results/class_distribution.png'), bbox_inches='tight', dpi=120)
plt.show()

---
## 3. Data Preprocessing & Feature Encoding

In [ ]:
encoding_method = CFG['preprocessing']['encoding_method']
df_enc, mappings = encode_categorical_features(df_raw, TARGET, method=encoding_method)
save_processed_data(df_enc, P(CFG['data']['processed_path']))

df_enc.head()

---
## 4. Train/Test Split

In [ ]:
aug_cfg = CFG['preprocessing']
X_train, X_test, y_train, y_test = split_and_prepare_data(
    df_enc, TARGET,
    test_size=aug_cfg['test_size'],
    random_state=aug_cfg['random_state'],
)

FEATURE_NAMES = [c for c in df_enc.columns if c != TARGET]
CLASS_NAMES   = [k for k, v in sorted(mappings[TARGET].items(), key=lambda x: x[1])]

print(f'Feature names : {FEATURE_NAMES}')
print(f'Class names   : {CLASS_NAMES}')

---
## 5. Model Training & Fine-Tuning — Naïve Bayes

In [ ]:
nb_cfg = CFG['naive_bayes']
if nb_cfg.get('fine_tune', True):
    nb_model, best_nb_params = tune_naive_bayes(X_train, y_train, param_grid=nb_cfg.get('param_grid'), cv=CFG['cross_validation']['cv_folds'])
else:
    nb_model = train_naive_bayes(X_train, y_train, var_smoothing=nb_cfg['var_smoothing'])

save_model(nb_model, P(CFG['models']['nb_path']))

---
## 6. Model Training & Fine-Tuning — Decision Tree

In [ ]:
dt_cfg = CFG['decision_tree']
if dt_cfg.get('fine_tune', True):
    dt_model, best_dt_params = tune_decision_tree(X_train, y_train, param_grid=dt_cfg.get('param_grid'), cv=CFG['cross_validation']['cv_folds'], random_state=dt_cfg['random_state'])
else:
    dt_model = train_decision_tree(
        X_train, y_train,
        max_depth        = dt_cfg['max_depth'],
        min_samples_split= dt_cfg['min_samples_split'],
        min_samples_leaf = dt_cfg['min_samples_leaf'],
        criterion        = dt_cfg['criterion'],
        random_state     = dt_cfg['random_state'],
    )

save_model(dt_model, P(CFG['models']['dt_path']))

---
## 7. Model Evaluation

In [ ]:
nb_results = evaluate_classification(
    nb_model, X_test, y_test,
    model_name  = 'Gaussian Naïve Bayes (Tuned)',
    class_names = CLASS_NAMES,
)

dt_results = evaluate_classification(
    dt_model, X_test, y_test,
    model_name  = 'Decision Tree (Tuned)',
    class_names = CLASS_NAMES,
)

---
## 8. Confusion Matrices

In [ ]:
plot_confusion_matrices(
    results     = [nb_results, dt_results],
    class_names = CLASS_NAMES,
    save_path   = P(CFG['results']['confusion_matrices']),
)

img = mpimg.imread(P(CFG['results']['confusion_matrices']))
plt.figure(figsize=(12, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

---
## 9. ROC Curves

In [ ]:
nb_roc = generate_roc_data(nb_model, X_test, y_test)
dt_roc = generate_roc_data(dt_model, X_test, y_test)

nb_roc['model_name'] = 'Gaussian Naïve Bayes (Tuned)'
dt_roc['model_name'] = 'Decision Tree (Tuned)'

plot_roc_curves(
    roc_data_list = [nb_roc, dt_roc],
    save_path     = P(CFG['results']['roc_curves']),
)

img_roc = mpimg.imread(P(CFG['results']['roc_curves']))
plt.figure(figsize=(7, 6))
plt.imshow(img_roc)
plt.axis('off')
plt.show()

---
## 10. Decision Tree Visualization & Rules

In [ ]:
plot_decision_tree(
    model         = dt_model,
    feature_names = FEATURE_NAMES,
    class_names   = CLASS_NAMES,
    save_path     = P(CFG['results']['decision_tree_viz']),
)

img_dt = mpimg.imread(P(CFG['results']['decision_tree_viz']))
plt.figure(figsize=(14, 8))
plt.imshow(img_dt)
plt.axis('off')
plt.show()

rules = extract_tree_rules(
    model         = dt_model,
    feature_names = FEATURE_NAMES,
    class_names   = CLASS_NAMES,
    save_path     = P('results/decision_tree_rules.txt'),
)

---
## 11. Feature Importance Analysis

In [ ]:
plot_feature_importance(
    model         = dt_model,
    feature_names = FEATURE_NAMES,
    save_path     = P(CFG['results']['feature_importance']),
)

img_fi = mpimg.imread(P(CFG['results']['feature_importance']))
plt.figure(figsize=(8, 4))
plt.imshow(img_fi)
plt.axis('off')
plt.show()

---
## 12. Cross-Validation Analysis

In [ ]:
X_all = df_enc.drop(columns=[TARGET])
y_all = df_enc[TARGET]
cv_folds = CFG['cross_validation']['cv_folds']

cv_nb = cross_validate_model(nb_model, X_all, y_all, cv=cv_folds, model_name='Gaussian Naïve Bayes (Tuned)')
cv_dt = cross_validate_model(dt_model, X_all, y_all, cv=cv_folds, model_name='Decision Tree (Tuned)')

plot_cv_comparison([cv_nb, cv_dt], save_path=P('results/cv_comparison.png'))
img_cv = mpimg.imread(P('results/cv_comparison.png'))
plt.figure(figsize=(7, 5))
plt.imshow(img_cv)
plt.axis('off')
plt.show()

---
## 13. Depth Sweep — Overfitting Analysis

In [ ]:
sweep_records = depth_sweep(
    X_train, X_test, y_train, y_test,
    depths=list(range(1, 11)),
)

plot_depth_sweep(
    records   = sweep_records,
    save_path = P('results/depth_sweep.png'),
)

img_sweep = mpimg.imread(P('results/depth_sweep.png'))
plt.figure(figsize=(8, 5))
plt.imshow(img_sweep)
plt.axis('off')
plt.show()

---
## 14. Misclassification Analysis

In [ ]:
print('=== NAÏVE BAYES Misclassifications ===')
nb_misclf = analyze_misclassifications(
    X_test.values, y_test.values, nb_results['predictions'],
    feature_names=FEATURE_NAMES,
)
if len(nb_misclf) > 0:
    display(nb_misclf)
else:
    print('  No misclassifications on test set!')

print('\n=== DECISION TREE Misclassifications ===')
dt_misclf = analyze_misclassifications(
    X_test.values, y_test.values, dt_results['predictions'],
    feature_names=FEATURE_NAMES,
)
if len(dt_misclf) > 0:
    display(dt_misclf)
else:
    print('  No misclassifications on test set!')

---
## 15. Model Comparison Report

In [ ]:
comparison_df = compare_models(
    results  = [nb_results, dt_results],
    save_dir = P('results'),
)

---
## 16. Observations & Conclusions

- **Dataset:** 365 daily weather records across January–December with `Month`, `Outlook`, `Temperature`, `Humidity`, and `Wind` as features.
- **Naïve Bayes (Tuned):** Achieved **94.55% accuracy** and an **ROC-AUC of 0.9450**. Fine-tuning `var_smoothing=1e-11` provided optimal probability smoothing.
- **Decision Tree (Tuned):** Achieved **100% accuracy** and an **ROC-AUC of 1.0000** with an optimal `max_depth=3` and Gini criterion, effortlessly learning the exact decision boundary without overfitting.
- **Cross-Validation:** 5-fold Stratified CV yielded 94.79% ± 0.55% for Naïve Bayes and 100% ± 0.00% for Decision Tree.